# JSONL LLM Trace Explorer

Interactive viewer for task-decomposition JSONL event logs. Run all cells, enter the JSONL file path when prompted, and the trace displays below. Supports any similarly structured JSONL file. No network access required.

**Install (only if needed):** `uv add --dev jupyterlab ipykernel notebook pandas`  
For VS Code, open this notebook with a Python/Jupyter kernel.

In [31]:
"""Interactive viewer for task-decomposition JSONL trace logs."""

import html
import json
from pathlib import Path

import pandas as pd
from IPython.display import display, HTML, Markdown

TRACE_PATH = Path(input('Enter the JSONL trace file path: ').strip()).expanduser()
if not TRACE_PATH.is_file():
    raise FileNotFoundError(f'Trace file not found: {TRACE_PATH}')


In [32]:
def load_trace(path):
    """Load a JSONL trace file into a list of event dicts plus parse issues."""
    parsed_events, parse_issues = [], []
    with Path(path).open(encoding='utf-8-sig') as f:
        for line_number, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                item = json.loads(line)
                if not isinstance(item, dict):
                    parse_issues.append(f'Line {line_number}: expected a JSON object')
                    continue
                item['_line'] = line_number
                parsed_events.append(item)
            except json.JSONDecodeError as exc:
                parse_issues.append(f'Line {line_number}: {exc}')
    return parsed_events, parse_issues


def stage_of(item):
    """Return the stage name for an event, or a session placeholder."""
    return item.get('stage') or '(session)'


def status_of(item):
    """Return a display status string for an event."""
    if item.get('result') is not None:
        return str(item['result'])
    if item.get('event') == 'execution.summary':
        return str(item.get('data', {}).get('status', '—'))
    return '—'


def as_table(records):
    """Build a flat one-row-per-event DataFrame for the trace summary."""
    table_rows = []
    for item in records:
        data = item.get('data', {})
        metadata = data.get('response_metadata', {})
        model = ''
        if isinstance(metadata, dict):
            model = data.get('model') or metadata.get('model', '')
        table_rows.append({
            'line': item['_line'],
            'timestamp': item.get('timestamp', ''),
            'stage': stage_of(item),
            'event': item.get('event', ''),
            'status': status_of(item),
            'latency_ms': data.get('latency_ms'),
            'model': model,
        })
    return pd.DataFrame(table_rows)


events, issues = load_trace(TRACE_PATH)
df = as_table(events)
print(f'Loaded {len(events)} events from {TRACE_PATH.name}; parse issues: {len(issues)}')
if issues:
    display(pd.DataFrame({'issue': issues}))


Loaded 22 events from task_decomposition_2026-10-17_014615.jsonl; parse issues: 0


## Execution summary

In [33]:
display(pd.DataFrame([{
    'events': len(events),
    'stages': df.loc[df.stage != '(session)', 'stage'].nunique(),
    'failures': int(df.status.isin(['failure', 'error', 'failed']).sum()),
    'total_llm_latency_s': round(
        pd.to_numeric(df.latency_ms, errors='coerce').sum() / 1000, 2
    ),
}]))
display(df.groupby('stage', sort=False).agg(
    events=('event', 'size'),
    last_status=('status', 'last'),
    llm_latency_ms=('latency_ms', 'sum'),
).reset_index())


def usage_rows(records):
    """Return one row per llm.response event with its numeric usage fields."""
    usage_records = []
    for item in records:
        usage_data = item.get('data', {}).get('usage')
        if item.get('event') != 'llm.response' or not isinstance(usage_data, dict):
            continue
        usage_records.append({
            'stage': stage_of(item),
            **{
                key: value
                for key, value in usage_data.items()
                if isinstance(value, (int, float))
            },
        })
    return pd.DataFrame(usage_records)


usage = usage_rows(events)
if not usage.empty:
    display(Markdown('### Token usage by response'))
    display(usage)


,events,stages,failures,total_llm_latency_s
0,22,4,0,11.28


,stage,events,last_status,llm_latency_ms
0,(session),1,—,0.000
1,operational_decomposition,5,success,5622.923
2,semantic_evaluation,4,accept,1405.055
3,retain_remove_classification,6,success,3281.129
4,added_work_classification,6,success,973.574


### Token usage by response

,stage,input_tokens,output_tokens,total_tokens
0,operational_decomposition,786,573,1359
1,semantic_evaluation,1928,37,1965
2,retain_remove_classification,1477,437,1914
3,added_work_classification,2008,51,2059


## Human-readable trace
Each JSONL event is shown below as a collapsible, formatted section.

In [34]:
def pretty(value):
    """Pretty-print JSON-encoded strings; return other values as JSON text."""
    if isinstance(value, str):
        try:
            value = json.loads(value)
        except (ValueError, TypeError):
            return value
    return json.dumps(value, ensure_ascii=False, indent=2, default=str)


def panel(title, value, open_by_default=False):
    """Render one collapsible HTML panel for a title and value pair."""
    content = html.escape(pretty(value))
    open_attr = 'open' if open_by_default else ''
    return (
        f'<details {open_attr}><summary style="cursor:pointer;font-weight:600;padding:7px">'
        f'{html.escape(title)}</summary>'
        f'<pre style="white-space:pre-wrap;overflow-wrap:anywhere;background:#f5f7fa;'
        f'color:#182230;padding:12px;border-radius:6px;max-height:520px;overflow:auto">'
        f'{content}</pre></details>'
    )


def show_event(item):
    """Render a single trace event as Markdown headings and HTML panels."""
    data = item.get('data') or {}
    title = (
        f"Line {item['_line']} · {item.get('event', '')}"
        f" · {stage_of(item)} · {status_of(item)}"
    )
    display(Markdown('### ' + title))
    sections = []
    if isinstance(data, dict):
        if 'messages' in data:
            for idx, msg in enumerate(data['messages']):
                sections.append(panel(
                    f"Message {idx + 1} — {msg.get('role', '')}",
                    msg.get('content', ''),
                ))
        for key in ('parsed_response', 'response', 'raw_output', 'errors', 'error',
                    'usage', 'parameters', 'structured_output_schema', 'raw_response'):
            if key in data:
                sections.append(
                    panel(key, data[key], key in ('errors', 'error', 'parsed_response'))
                )
        remaining = {
            key: value for key, value in data.items()
            if key not in ('messages', 'parsed_response', 'response', 'raw_output',
                           'errors', 'error', 'usage', 'parameters',
                           'structured_output_schema', 'raw_response')
        }
        if remaining:
            sections.append(panel('Other data', remaining))
    sections.append(panel('Complete event JSON', item))
    display(HTML(''.join(sections)))


for event in events:
    show_event(event)


### Line 1 · trace.session · (session) · —

### Line 2 · llm.request · operational_decomposition · —

### Line 3 · llm.response · operational_decomposition · success

### Line 4 · llm.parsed · operational_decomposition · —

### Line 5 · validation.structural · operational_decomposition · success

...

### Line 18 · llm.response · added_work_classification · success

### Line 19 · llm.parsed · added_work_classification · —

### Line 20 · validation.structural · added_work_classification · success

### Line 21 · validation.structural · added_work_classification · success

### Line 22 · execution.summary · added_work_classification · success

## Structured output: operational subtasks
Extracts the decomposition from parsed events, validation results, or response text, preferring parsed output.

In [35]:
def find_subtasks(obj):
    """Recursively find the first object that has 'operational_subtasks'."""
    if isinstance(obj, dict):
        subtask_list = obj.get('operational_subtasks')
        if isinstance(subtask_list, list):
            return obj
        for value in obj.values():
            result = find_subtasks(value)
            if result is not None:
                return result
    elif isinstance(obj, list):
        for value in obj:
            result = find_subtasks(value)
            if result is not None:
                return result
    elif isinstance(obj, str):
        try:
            return find_subtasks(json.loads(obj))
        except (ValueError, TypeError):
            pass
    return None


candidates = sorted(events, key=lambda item: 0 if item.get('event') == 'llm.parsed' else 1)
decomposition = next(
    (found_subtasks for item in candidates
     if (found_subtasks := find_subtasks(item.get('data', {}))) is not None),
    None,
)
if decomposition:
    task = decomposition.get('task', {})
    display(Markdown('**Task:** ' + str(task.get('task_name', 'Unknown'))))
    subtasks = pd.DataFrame(decomposition['operational_subtasks'])
    if 'depends_on' in subtasks:
        subtasks['depends_on'] = subtasks['depends_on'].apply(
            lambda value: ', '.join(map(str, value)) if isinstance(value, list) else str(value)
        )
    display(subtasks.drop(columns=['provenance_refs'], errors='ignore').style.set_properties(
        **{'text-align': 'left', 'white-space': 'pre-wrap'}
    ))
else:
    print('No operational_subtasks object found in this trace.')


**Task:** Production Scheduling

,sequence_index,subtask_id,subtask_name,description,relative_effort_weight,depends_on
0,1,subtask-1,Gather Production Requirements,Collect information on customer orders and production requirements.,0.15,
1,2,subtask-2,Assess Resource Availability,"Evaluate available resources, including materials, equipment, and workforce for production.",0.204,subtask-1
2,3,subtask-3,Identify Operational Constraints,Analyze and document any operational constraints that may impact production.,0.105,subtask-2
3,4,subtask-4,Create Production Schedule,"Develop a detailed production schedule based on the gathered requirements, resource availability, and constraints.",0.305,subtask-3
4,5,subtask-5,Communicate Schedule to Teams,Distribute the production schedule to relevant teams for execution.,0.10192,subtask-4
5,6,subtask-6,Monitor Production Progress,Track production progress against the schedule and troubleshoot any issues.,0.20005,subtask-5


## RETAIN / REMOVE classification
Shows which operational subtasks remain human-performed and which are automated or delegated.

In [36]:
def find_classification(obj):
    """Recursively find the first object that has 'classified_subtasks'."""
    if isinstance(obj, dict):
        classified_rows = obj.get('classified_subtasks')
        if isinstance(classified_rows, list):
            return obj
        for value in obj.values():
            found = find_classification(value)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for value in obj:
            found = find_classification(value)
            if found is not None:
                return found
    elif isinstance(obj, str):
        try:
            return find_classification(json.loads(obj))
        except (ValueError, TypeError):
            pass
    return None


classification = next(
    (found_classification for item in events
     if (found_classification := find_classification(item.get('data', {}))) is not None),
    None,
)
if classification:
    names = {
        row.get('subtask_id'): row.get('subtask_name')
        for row in decomposition.get('operational_subtasks', [])
    } if decomposition else {}
    rows = []
    for row in classification['classified_subtasks']:
        decision = str(row.get('classification', '')).upper()
        rows.append({
            'subtask_id': row.get('subtask_id'),
            'subtask_name': row.get('subtask_name') or names.get(row.get('subtask_id'), ''),
            'status': decision,
            'human work retained': 'Yes' if decision == 'RETAIN' else 'No',
            'automated/delegated': 'Yes' if decision == 'REMOVE' else 'No',
        })
    display(pd.DataFrame(rows).style.set_properties(
        **{'text-align': 'left', 'white-space': 'pre-wrap'}
    ))
else:
    print('No RETAIN/REMOVE classification found in this trace.')


,subtask_id,subtask_name,status,human work retained,automated/delegated
0,subtask-1,Gather Production Requirements,RETAIN,Yes,No
1,subtask-2,Assess Resource Availability,RETAIN,Yes,No
2,subtask-3,Identify Operational Constraints,RETAIN,Yes,No
3,subtask-4,Create Production Schedule,REMOVE,No,Yes
4,subtask-5,Communicate Schedule to Teams,RETAIN,Yes,No
5,subtask-6,Monitor Production Progress,RETAIN,Yes,No


## Validation diagnostics

In [37]:
diagnostics = []
for event in events:
    if (
        event.get('event', '').startswith('validation.')
        or event.get('event') == 'execution.summary'
    ):
        event_data = event.get('data') or {}
        diagnostics.append({
            'line': event['_line'],
            'stage': stage_of(event),
            'event': event.get('event'),
            'result': status_of(event),
            'errors': pretty(event_data.get('errors', event_data.get('error', '')))[:2500],
        })
if diagnostics:
    display(pd.DataFrame(diagnostics).style.set_properties(
        **{'white-space': 'pre-wrap', 'text-align': 'left'}
    ))
else:
    print('No validation events found.')


,line,stage,event,result,errors
0,5,operational_decomposition,validation.structural,success,[]
1,6,operational_decomposition,validation.structural,success,[]
2,10,semantic_evaluation,validation.evaluation,accept,[]
3,14,retain_remove_classification,validation.structural,success,[]
4,15,retain_remove_classification,validation.structural,success,[]
5,16,retain_remove_classification,validation.structural,success,[]
6,20,added_work_classification,validation.structural,success,[]
7,21,added_work_classification,validation.structural,success,[]
8,22,added_work_classification,execution.summary,success,
